# Notebook 04: Transaction Fraud Models

Evaluate the merged scripted behavioural transactions and PaySim in separate experiments. Their feature sets and fraud-label meanings differ, so never merge their rows or scores into one model.

## Experiment A: merged scripted behavioural data

These results measure whether the prototype learns the generated behavioural scenarios. They are not PaySim or real-world performance.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.behaviour_detector import BEHAVIOURAL_COLUMNS, build_behaviour_model, build_anomaly_model

transactions = pd.read_csv(PROJECT_ROOT / "data" / "synthetic" / "transactions.csv")
X = transactions[BEHAVIOURAL_COLUMNS]
y = transactions["is_fraudulent"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42,
)
classifier = build_behaviour_model()
classifier.fit(X_train, y_train)
predictions = classifier.predict(X_test)
probabilities = classifier.predict_proba(X_test)[:, 1]

print(classification_report(y_test, predictions, target_names=["legitimate", "fraud"]))
print("Confusion matrix:", confusion_matrix(y_test, predictions))
print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))
print("PR-AUC:", round(average_precision_score(y_test, probabilities), 4))
print("Feature importances:")
display(dict(sorted(zip(BEHAVIOURAL_COLUMNS, classifier.feature_importances_), key=lambda item: item[1], reverse=True)))

              precision    recall  f1-score   support

  legitimate       0.99      0.97      0.98       375
       fraud       0.91      0.96      0.93       125

    accuracy                           0.97       500
   macro avg       0.95      0.96      0.96       500
weighted avg       0.97      0.97      0.97       500

Confusion matrix: [[363  12]
 [  5 120]]
ROC-AUC: 0.9901
PR-AUC: 0.9761
Feature importances:


{'recent_suspicious_sms': np.float64(0.2647934494853165),
 'credential_request_detected': np.float64(0.24452717539169122),
 'amount': np.float64(0.20209951992257724),
 'recent_suspicious_call': np.float64(0.2007513164832275),
 'is_new_recipient': np.float64(0.0721018941666402),
 'new_device': np.float64(0.015726644550547304)}

In [2]:
anomaly_model = build_anomaly_model()
anomaly_model.fit(X)
anomaly_predictions = anomaly_model.predict(X)
display(transactions.assign(anomaly=anomaly_predictions)[
    ["transaction_id", "amount", "is_fraudulent", "anomaly"]
].head(10))

,transaction_id,amount,is_fraudulent,anomaly
0,TXN00764,1407.0,1,-1
1,TXN00359,2449.0,1,-1
2,TXN00092,4897.0,1,-1
3,TXN00765,2750.0,1,-1
4,TXN00896,567.0,0,1
5,TXN00677,1072.0,1,-1
6,TXN00842,4624.0,1,-1
7,TXN00297,270.0,0,1
8,TXN00243,689.0,0,-1
9,TXN00811,1718.0,1,-1


## Experiment B: PaySim transaction fraud

Use an out-of-time split (`step <= 354` for training, later steps for testing), and a reproducible 20% row sample within each period to keep execution tractable. Only pre-transaction fields `step`, `type`, and `amount` are used. Account balances and the existing `isFlaggedFraud` flag are excluded to avoid known leakage.

In [3]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

paysim_path = PROJECT_ROOT / "data" / "raw" / "PS_20174392719_1491204439457_log.csv"
if not paysim_path.exists():
    raise FileNotFoundError(f"PaySim CSV not found: {paysim_path}")

train_parts, test_parts = [], []
time_cutoff = 354
sample_fraction = 0.20
for chunk_index, chunk in enumerate(pd.read_csv(
    paysim_path,
    usecols=["step", "type", "amount", "isFraud"],
    chunksize=500_000,
)):
    train_mask = chunk["step"] <= time_cutoff
    for mask, destination in ((train_mask, train_parts), (~train_mask, test_parts)):
        period = chunk.loc[mask]
        if not period.empty:
            destination.append(period.sample(frac=sample_fraction, random_state=42 + chunk_index))

paysim_train = pd.concat(train_parts, ignore_index=True)
paysim_test = pd.concat(test_parts, ignore_index=True)
transaction_types = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]

def paysim_features(frame):
    features = pd.DataFrame({
        "step": frame["step"].astype("float32"),
        "log_amount": np.log1p(frame["amount"].clip(lower=0).astype("float32")),
    })
    for transaction_type in transaction_types:
        features[f"type_{transaction_type}"] = frame["type"].eq(transaction_type).astype("int8")
    return features

X_train = paysim_features(paysim_train)
X_test = paysim_features(paysim_test)
y_train = paysim_train["isFraud"].astype("int8")
y_test = paysim_test["isFraud"].astype("int8")

paysim_model = HistGradientBoostingClassifier(
    max_iter=75,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    early_stopping=True,
    random_state=42,
)
paysim_model.fit(
    X_train,
    y_train,
    sample_weight=compute_sample_weight("balanced", y_train),
)
paysim_probabilities = paysim_model.predict_proba(X_test)[:, 1]
paysim_predictions = (paysim_probabilities >= 0.5).astype("int8")

print("Training sample:", X_train.shape, "fraud:", int(y_train.sum()))
print("Later-time test sample:", X_test.shape, "fraud:", int(y_test.sum()))
print("Test fraud prevalence / no-skill PR-AUC:", round(float(y_test.mean()), 5))
print("ROC-AUC:", round(roc_auc_score(y_test, paysim_probabilities), 4))
print("PR-AUC:", round(average_precision_score(y_test, paysim_probabilities), 4))
print("Confusion matrix at threshold 0.5:", confusion_matrix(y_test, paysim_predictions))
print(classification_report(y_test, paysim_predictions, target_names=["legitimate", "fraud"]))

Training sample: (1013819, 7) fraud: 752
Later-time test sample: (258705, 7) fraud: 822
Test fraud prevalence / no-skill PR-AUC: 0.00318
ROC-AUC: 0.9139
PR-AUC: 0.2376


Confusion matrix at threshold 0.5: [[235718  22165]
 [   245    577]]
              precision    recall  f1-score   support

  legitimate       1.00      0.91      0.95    257883
       fraud       0.03      0.70      0.05       822

    accuracy                           0.91    258705
   macro avg       0.51      0.81      0.50    258705
weighted avg       1.00      0.91      0.95    258705

